# Proyecto final — Informe de solución

## 1. Objetivo y problema de negocio

Interconnect busca reducir la pérdida de clientes mediante la identificación anticipada de usuarios con mayor riesgo de cancelar sus servicios.

El objetivo del proyecto fue desarrollar un modelo de clasificación binaria capaz de estimar la probabilidad de churn de cada cliente a partir de información contractual, demográfica y de los servicios contratados.

La utilidad empresarial del modelo consiste en proporcionar al área de Marketing un **score de riesgo de cancelación**, permitiendo priorizar clientes para acciones preventivas de retención, como promociones, ofertas especiales o modificaciones de plan.

La variable objetivo se definió como:

- `Churn = 1`: el cliente canceló el servicio.
- `Churn = 0`: el cliente permanece activo.

Se utilizó **AUC-ROC como métrica principal** y Accuracy como métrica adicional.

---

## 2. Datos y metodología

Los datos se encontraban distribuidos en cuatro fuentes principales:

- información contractual;
- información personal;
- servicios de Internet;
- servicios telefónicos.

Las tablas se integraron utilizando `customerID` como identificador único de cliente.

El desarrollo siguió las siguientes etapas:

1. comprensión del problema de negocio;
2. auditoría inicial de los datos;
3. integración de las fuentes;
4. limpieza y transformación;
5. construcción de la variable objetivo;
6. análisis exploratorio;
7. ingeniería de características;
8. separación de entrenamiento y prueba;
9. construcción del pipeline de preprocesamiento;
10. establecimiento de modelos baseline;
11. comparación de algoritmos;
12. optimización de hiperparámetros;
13. evaluación final;
14. interpretabilidad;
15. conclusiones y recomendaciones de negocio.

El conjunto de datos se dividió en entrenamiento y prueba utilizando estratificación para conservar aproximadamente la proporción original de churn.

El conjunto de prueba permaneció aislado durante la comparación de modelos y la optimización de hiperparámetros.

Para la evaluación durante el desarrollo se utilizó validación cruzada estratificada de cinco folds.

### Pasos modificados u omitidos

No se omitieron etapas fundamentales del plan de trabajo, aunque algunas decisiones fueron modificadas a partir de los resultados obtenidos durante el análisis.

Se construyó inicialmente una variable denominada `HistoricalTenure` para representar la antigüedad histórica del cliente. Sin embargo, no se incorporó al modelo principal porque, para los clientes que cancelaron, dependía de su fecha real de cancelación (`EndDate`).

Utilizar esta información en un escenario predictivo introduciría **fuga de información temporal (`data leakage`)**, ya que dicha fecha no estaría disponible en el momento real de realizar una predicción.

Por esta razón se decidió excluir esta variable del modelo final, priorizando una evaluación metodológicamente más realista.

También se realizó un experimento adicional con CatBoost utilizando directamente variables categóricas. Esta variante mejoró el baseline inicial, aunque después de la optimización obtuvo un resultado prácticamente equivalente al modelo con One-Hot Encoding.

---

## 3. Modelo seleccionado

Se evaluaron diferentes familias de algoritmos:

- `DummyClassifier`;
- `LogisticRegression`;
- `RandomForestClassifier`;
- `GradientBoostingClassifier`;
- `CatBoostClassifier`.

El `DummyClassifier` se utilizó como referencia mínima y obtuvo un AUC-ROC de aproximadamente `0.50`, indicando ausencia de capacidad discriminativa.

La Regresión Logística proporcionó un baseline competitivo:

```text
ROC-AUC ≈ 0.8399
Accuracy ≈ 0.8007

## 4. Resultados finales

Durante la comparación inicial de modelos, **Gradient Boosting** obtuvo el mayor AUC-ROC promedio:

| Modelo | AUC-ROC promedio |
|---|---:|
| Gradient Boosting | 0.8477 |
| CatBoost | 0.8448 |
| Logistic Regression | 0.8399 |
| Random Forest | 0.8182 |
| Dummy Classifier | 0.5000 |

Posteriormente, se realizó una optimización de hiperparámetros. Los mejores resultados fueron:

| Modelo optimizado | ROC-AUC |
|---|---:|
| CatBoost OHE Tuned | ≈ 0.850601 |
| Gradient Boosting Tuned | ≈ 0.850466 |
| CatBoost Native Tuned | ≈ 0.850246 |

Las diferencias entre los tres modelos fueron pequeñas. Sin embargo, de acuerdo con el criterio establecido previamente de seleccionar el mayor AUC-ROC promedio, se seleccionó como modelo final el **CatBoostClassifier optimizado con One-Hot Encoding**.

El modelo seleccionado fue evaluado una única vez sobre el conjunto de prueba, que permaneció separado durante todo el proceso de desarrollo.

| Métrica | Resultado |
|---|---:|
| AUC-ROC | 0.843972 |
| Accuracy | 0.807665 |
| Precision — Churn | 0.67 |
| Recall — Churn | 0.53 |
| F1-score — Churn | 0.60 |

Durante la validación cruzada, el modelo había obtenido:

```text
ROC-AUC CV   ≈ 0.850601
ROC-AUC Test ≈ 0.843972
```

La diferencia fue aproximadamente:

```text
-0.00663
```

Esta reducción es pequeña y sugiere que el modelo mantiene un comportamiento razonablemente consistente sobre datos no utilizados durante el desarrollo.

La matriz de confusión fue:

|  | Predicción: No Churn | Predicción: Churn |
|---|---:|---:|
| Real: No Churn | 938 | 97 |
| Real: Churn | 174 | 200 |

Esto significa que:

- 938 clientes activos fueron correctamente identificados.
- 97 clientes activos fueron clasificados como posibles casos de churn.
- 200 clientes que cancelaron fueron correctamente detectados.
- 174 clientes que cancelaron no fueron detectados por el modelo.

El principal punto de mejora se encuentra en el **recall de churn**. Utilizando el umbral estándar de clasificación, el modelo detecta aproximadamente el 53 % de las cancelaciones reales.

## 5. Principales hallazgos

El análisis exploratorio y las técnicas de interpretabilidad mostraron resultados consistentes. Las variables con mayor relevancia predictiva fueron:

- `Type`.
- `TotalCharges`.
- `InternetService`.
- `PaymentMethod`.
- `MonthlyCharges`.
- `TechSupport`.
- `OnlineSecurity`.

La importancia agregada calculada por CatBoost identificó principalmente:

| Variable | Importancia aproximada |
|---|---:|
| `Type` | 28.12 |
| `TotalCharges` | 21.52 |
| `InternetService` | 14.05 |
| `MonthlyCharges` | 6.86 |
| `OnlineSecurity` | 4.49 |
| `TechSupport` | 4.28 |
| `PaymentMethod` | 4.24 |

El análisis mediante SHAP confirmó un patrón similar:

| Variable | Importancia SHAP aproximada |
|---|---:|
| `Type` | 0.644 |
| `TotalCharges` | 0.525 |
| `InternetService` | 0.409 |
| `PaymentMethod` | 0.214 |
| `MonthlyCharges` | 0.209 |
| `TechSupport` | 0.190 |
| `OnlineSecurity` | 0.183 |

La coincidencia entre ambos métodos aumenta la confianza en que el modelo utiliza de forma consistente estas características para generar sus predicciones.

El tipo de contrato fue la característica dominante. Durante el análisis exploratorio también se observaron diferencias importantes en el churn entre los contratos mensuales y los contratos de mayor duración.

`TotalCharges` presentó una alta relevancia predictiva y puede reflejar parcialmente diferencias relacionadas con la trayectoria y la permanencia acumulada del cliente.

También se observaron patrones relevantes asociados con:

- Tipo de servicio de Internet.
- Método de pago.
- Cargos mensuales.
- Soporte técnico.
- Seguridad en línea.

Estos resultados representan asociaciones predictivas, no relaciones causales. Por lo tanto, no puede concluirse que modificar directamente una de estas características produzca automáticamente una reducción del churn.

## 6. Dificultades y limitaciones

Una de las principales dificultades fue construir una representación temporal válida de la antigüedad del cliente.

La información disponible permite conocer la fecha de inicio del contrato y, para los clientes que ya cancelaron, su fecha de cancelación. Sin embargo, utilizar esta última información para calcular la antigüedad dentro del modelo predictivo introduciría información que no estaría disponible antes de que ocurriera el evento.

Por esta razón, `HistoricalTenure` fue excluida del modelo principal.

Otra dificultad se encontró en `TotalCharges`, que originalmente estaba almacenada como texto debido a la existencia de valores vacíos, correspondientes principalmente a clientes recientes. Después de analizar estos registros, la variable se convirtió a formato numérico y los casos correspondientes se trataron de acuerdo con su contexto contractual.

Los valores ausentes provenientes de las tablas de Internet y telefonía tampoco representaban necesariamente datos desconocidos. En muchos casos indicaban que el cliente simplemente no tenía contratado ese servicio. Por ello, se codificaron explícitamente como ausencia de servicio.

También fue fundamental evitar el uso de información del conjunto de prueba durante:

- La selección de variables.
- La comparación de algoritmos.
- La optimización de hiperparámetros.
- La selección del modelo final.

Entre las principales limitaciones del proyecto se encuentran:

- AUC-ROC final inferior a 0.88.
- Recall moderado para la clase churn.
- Ausencia de variables de satisfacción del cliente.
- Ausencia de información sobre reclamaciones.
- Falta de datos sobre interrupciones o calidad del servicio.
- Falta de historial de contactos con soporte.
- Ausencia de información detallada sobre comportamiento de uso.
- Ausencia de snapshots temporales que permitan reconstruir el estado histórico del cliente.

Estas variables podrían proporcionar señal adicional en futuras versiones del modelo.

## 7. Conclusiones

Los pasos clave para resolver la tarea fueron:

- Integrar correctamente las diferentes fuentes de datos.
- Comprender el significado de los valores faltantes.
- Construir una variable objetivo coherente con el problema de negocio.
- Prevenir la fuga de información.
- Realizar análisis exploratorio antes de diseñar características.
- Establecer un baseline antes de utilizar modelos complejos.
- Evaluar los algoritmos bajo la misma estrategia de validación.
- Optimizar únicamente los candidatos con mejor desempeño.
- Mantener el conjunto de prueba completamente aislado.
- Interpretar el modelo antes de formular recomendaciones de negocio.

Un resultado especialmente relevante fue comprobar que una mayor complejidad no garantiza automáticamente un mejor modelo. Random Forest obtuvo un AUC inferior al de Regresión Logística, mientras que Gradient Boosting y CatBoost produjeron los mejores resultados.

El modelo final obtuvo:

```text
AUC-ROC Test ≈ 0.8440
Accuracy     ≈ 0.8077
```

Por tanto, presenta una capacidad predictiva útil, aunque todavía existe margen de mejora.

La principal utilidad del modelo no consiste únicamente en producir una clasificación binaria. Su valor también está en generar una probabilidad que permita ordenar a los clientes según su riesgo estimado de cancelación.

## 8. Recomendaciones

Se recomienda utilizar el modelo como herramienta de apoyo para priorizar clientes dentro de campañas de retención.

En lugar de aplicar acciones comerciales a toda la cartera, Interconnect podría calcular periódicamente la probabilidad de churn de cada cliente activo y utilizarla para generar un ranking de riesgo.

Los clientes con mayores probabilidades podrían analizarse mediante estrategias diferenciadas según su perfil. Entre las posibles líneas de intervención se encuentran:

- Analizar clientes con contratos de corta duración.
- Evaluar alternativas contractuales de mayor duración.
- Revisar planes para segmentos con cargos mensuales relativamente elevados.
- Evaluar promociones relacionadas con soporte técnico o seguridad en línea.
- Analizar los métodos de pago y posibles incentivos hacia esquemas automáticos.
- Diseñar campañas diferenciadas según el score y las características del cliente.

No se recomienda utilizar una única promoción para todos los clientes clasificados como de alto riesgo. La probabilidad de churn debería combinarse con la información del perfil del cliente y con las restricciones comerciales disponibles.

También se recomienda analizar el umbral de clasificación desde una perspectiva económica. Reducirlo podría incrementar el recall y permitir detectar más cancelaciones, pero también aumentaría los falsos positivos y el costo de las campañas.

La selección del umbral debería considerar:

- Costo de una acción de retención.
- Valor esperado de conservar al cliente.
- Costo de adquisición de un nuevo cliente.
- Pérdida económica asociada al churn.
- Presupuesto disponible.

Las acciones comerciales propuestas deben considerarse hipótesis derivadas de asociaciones predictivas. Para determinar si realmente reducen el churn, deberían validarse mediante experimentos controlados, como pruebas A/B.

Para futuras versiones, se recomienda además:

- Incorporar nuevas fuentes de información.
- Generar snapshots temporales.
- Desarrollar variables de comportamiento reciente.
- Analizar la calibración de las probabilidades.
- Monitorear el drift de los datos.
- Medir el rendimiento del modelo después del despliegue.
- Establecer procesos de reentrenamiento periódico.

En un escenario productivo, el flujo podría seguir la siguiente estructura:

```text
Clientes activos
      ↓
Preparación de variables
      ↓
Modelo de churn
      ↓
Probabilidad de cancelación
      ↓
Ranking de riesgo
      ↓
Segmentación
      ↓
Acciones de retención
      ↓
Medición de resultados
```

De esta forma, el modelo puede convertirse en una herramienta de apoyo para dirigir los recursos de retención hacia los clientes con mayor riesgo estimado de cancelación.